# Induction Head Test

What are the top binding heads *doing*? — Pythia-2.8B head characterization

Does the compound "binding" signal reflect concept representation, or mundane head
types (induction, previous-token, attention-sink)? Heads are **derived from**
`results/binding/pythia/pythia-2.8b/pythia-2.8b-accessibility.csv` (never hardcoded — indices are
architecture-specific). Logic lives in `src/head_characterization.py`; written-up
findings in `docs/findings/head-characterization-findings.md`.

## Setup

In [1]:
# Cell 00: Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [2]:
# Cell 0: Imports
import sys
import torch
from pathlib import Path

# Colab: files are in /content/
# Local: notebook is in notebooks/, project root is one level up
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


In [3]:
# Cell 1: Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [ ]:
model_name = "pythia-2.8b"

model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 1. Derive top binding heads (from the binding CSV, not hardcoded)

In [7]:
# --- Derive top binding heads from the binding sweep (NOT hardcoded) ---
# Head indices are architecture-specific: GPT-2 XL's L15/H19 is a strong binder,
# but in Pythia-2.8B that same index has ~zero binding. So the head set must come
# from THIS model's binding CSV, never a carried-over literal.
from src.head_characterization import (
    get_top_binding_heads, characterize_heads, collocation_scan,
    heads_as_tuples, save_head_results,
)

BINDING_CSV = (PROJECT_ROOT / "results" / "binding" / "pythia" / model_name
               / f"{model_name}-accessibility.csv")
assert BINDING_CSV.exists(), f"binding CSV not found: {BINDING_CSV}"

top_all  = get_top_binding_heads(BINDING_CSV, n_per_compound=1)                # incl. early/positional
top_late = get_top_binding_heads(BINDING_CSV, n_per_compound=1, min_layer=10)  # non-positional candidates

print("Top binding heads — ALL layers:")
print(top_all.to_string(index=False))
print("\nTop binding heads — LATE layers (>=10):")
print(top_late.to_string(index=False))

heads_to_test = sorted(set(heads_as_tuples(top_all)) | set(heads_as_tuples(top_late)))
late_heads = heads_as_tuples(top_late)
print("\nHeads to characterize:", heads_to_test)

Top binding heads — ALL layers:
 layer  head  mean_binding  max_binding  n_compounds_top                                                                                    top_compounds
     1    12        0.9594       0.9913                7 color_contrast, form_label, keyboard_navigation, link_text, page_title, screen_reader, skip_link
     3     1        0.7862       0.9939                1                                                                                         alt_text
    30    29        0.6788       0.9920                1                                                                                    semantic_html
    27    24        0.6576       0.9765                1                                                                                  closed_captions
    10    16        0.5762       0.9823                1                                                                                  focus_indicator

Top binding heads — LATE layers (>=10):
 la

## 2. Token-pattern battery — induction / previous-token / duplicate-token

In [8]:
# Token-pattern battery: induction / previous-token / duplicate-token.
# One repeated-random-sequence forward pass; heads are the derived set above.
# (Replaces the old hardcoded GPT-2 head list + inline induction loop.)
char = characterize_heads(model, heads_to_test, seq_len=50, seed=0)
print(char.sort_values(["layer", "head"]).to_string(index=False))

 layer  head  induction  prev_token  dup_token               type
     1    12     0.0000      0.8860     0.0000     previous-token
     3     1     0.0141      0.2737     0.0126 partial (prev/dup)
    10    14     0.0004      0.3160     0.0097 partial (prev/dup)
    10    16     0.0116      0.0607     0.0169    uncharacterized
    27    24     0.0193      0.0502     0.0181    uncharacterized
    28    15     0.0127      0.0572     0.0147    uncharacterized
    29     7     0.0201      0.0216     0.0213    uncharacterized
    30    29     0.0115      0.0690     0.0028    uncharacterized


## 3. Verdict: are they induction heads?

In [9]:
# Verdict on the induction hypothesis + head-type breakdown.
# Induction heads score >= 0.5; the 2026-06-14 finding predicts NONE here.
induction_hits = char[char.induction >= 0.5][["layer", "head", "induction"]].values.tolist()
print("Induction heads (>=0.5):", induction_hits or "NONE — not induction heads")
print(f"Max induction score across tested heads: {char.induction.max():.4f}")
print("\nHead-type counts:")
print(char["type"].value_counts().to_string())
print("\nLate-layer (non-positional) heads — the 'uncharacterized' candidates:")
print(char[char.layer >= 10].sort_values("layer").to_string(index=False))

Induction heads (>=0.5): NONE — not induction heads
Max induction score across tested heads: 0.0201

Head-type counts:
type
uncharacterized       5
partial (prev/dup)    2
previous-token        1

Late-layer (non-positional) heads — the 'uncharacterized' candidates:
 layer  head  induction  prev_token  dup_token               type
    10    14     0.0004      0.3160     0.0097 partial (prev/dup)
    10    16     0.0116      0.0607     0.0169    uncharacterized
    27    24     0.0193      0.0502     0.0181    uncharacterized
    28    15     0.0127      0.0572     0.0147    uncharacterized
    29     7     0.0201      0.0216     0.0213    uncharacterized
    30    29     0.0115      0.0690     0.0028    uncharacterized


## 4. Raw attention patterns — what do the late heads attend to?

In [10]:
# Raw attention dump for the late-layer (non-positional) heads, to eyeball what
# they attend to. mystery_heads is derived, not hardcoded.
prompts = [
    "A screen reader is",
    "A bicycle wheel is",
    "The purpose of alt text is to",
]

mystery_heads = late_heads  # the late-layer candidates derived above

for prompt in prompts:
    tokens = model.to_tokens(prompt)
    str_tokens = model.to_str_tokens(prompt)
    _, cache = model.run_with_cache(tokens)

    print(f"\n{'='*50}")
    print(f"Prompt: {prompt}")
    print(f"Tokens: {str_tokens}")

    for layer, head in mystery_heads:
        attn = cache["pattern", layer][0, head]
        print(f"\nL{layer}/H{head}:")
        for i, tok in enumerate(str_tokens):
            weights = [f"{attn[i,j]:.3f}" for j in range(len(str_tokens))]
            print(f"  {tok:>12} attends to: {weights}")


Prompt: A screen reader is
Tokens: ['<|endoftext|>', 'A', ' screen', ' reader', ' is']

L30/H29:
  <|endoftext|> attends to: ['1.000', '0.000', '0.000', '0.000', '0.000']
             A attends to: ['1.000', '0.000', '0.000', '0.000', '0.000']
        screen attends to: ['0.994', '0.006', '0.000', '0.000', '0.000']
        reader attends to: ['0.821', '0.016', '0.002', '0.161', '0.000']
            is attends to: ['0.005', '0.000', '0.000', '0.981', '0.014']

L27/H24:
  <|endoftext|> attends to: ['1.000', '0.000', '0.000', '0.000', '0.000']
             A attends to: ['0.000', '1.000', '0.000', '0.000', '0.000']
        screen attends to: ['0.000', '0.908', '0.092', '0.000', '0.000']
        reader attends to: ['0.003', '0.882', '0.103', '0.013', '0.000']
            is attends to: ['0.035', '0.811', '0.121', '0.032', '0.001']

L10/H16:
  <|endoftext|> attends to: ['1.000', '0.000', '0.000', '0.000', '0.000']
             A attends to: ['0.984', '0.016', '0.000', '0.000', '0.000']
   

## 5. Attention-sink (BOS) check

In [ ]:
# Structural signals + final reclassification. Most "uncharacterized" late heads
# either park attention on <|endoftext|> (BOS sink) or on position 1 (the A/The
# token, a structural head). Fold both into the type label so nothing stays
# "uncharacterized" without reason.
from src.head_characterization import attention_to_bos, attention_to_position, final_label

bos  = attention_to_bos(model, heads_to_test)
pos1 = (attention_to_position(model, heads_to_test, position=1)
        .rename(columns={"attn_to_pos1": "pos1_attention"}))

char_bos = (char.merge(bos, on=["layer", "head"], how="left")
                .merge(pos1, on=["layer", "head"], how="left"))
char_bos["type"] = char_bos.apply(
    lambda r: final_label(r.induction, r.prev_token, r.dup_token,
                          r.bos_attention, r.pos1_attention),
    axis=1,
)
print(char_bos.sort_values("layer").to_string(index=False))

print("\nFinal type counts:")
print(char_bos["type"].value_counts().to_string())

## 6. Cross-domain collocation scan (uniform template)

In [12]:
# Is each late head accessibility-specific, domain-general, or just lexical?
# UNIFORM template for every compound removes the prompt-position confound
# (e.g. "Due process is" puts word1 at position 1, where L27/H24 already looks).
colloc = collocation_scan(model, late_heads, template="A {w1} {w2} is")

pivot = (colloc.groupby(["layer", "head", "domain"])["score"]
               .mean().unstack("domain"))
cols = [c for c in ["a11y", "finance", "legal", "medical", "general"] if c in pivot.columns]
print("Mean word2->word1 attention by domain (uniform template):")
print(pivot[cols].round(3).to_string())

print("\nPer-compound detail:")
print(colloc.sort_values(["layer", "head", "score"], ascending=[True, True, False]).to_string(index=False))

Mean word2->word1 attention by domain (uniform template):
domain       a11y  finance  legal  medical  general
layer head                                         
10    14    0.437    0.116  0.292    0.190    0.237
      16    0.331    0.074  0.020    0.023    0.069
27    24    0.058    0.216  0.012    0.002    0.069
28    15    0.260    0.663  0.144    0.116    0.168
29    7     0.198    0.021  0.018    0.018    0.077
30    29    0.402    0.046  0.053    0.004    0.340

Per-compound detail:
 layer  head  domain        compound  score
    10    14   legal     court order 0.7090
    10    14    a11y        alt text 0.6327
    10    14    a11y  color contrast 0.6252
    10    14 medical      heart rate 0.4274
    10    14 general   bicycle wheel 0.3818
    10    14    a11y       skip link 0.3505
    10    14    a11y focus indicator 0.2934
    10    14    a11y   screen reader 0.2828
    10    14 general    coffee table 0.2598
    10    14 finance    credit score 0.1746
    10    14 finance

## 7. Save results

In [13]:
# Save to results/adhoc/head_characterization/ (so the run never has to be repeated).
# Persists the BOS column too if the attention-sink cell has been run.
save_head_results(model_name, PROJECT_ROOT,
                  char_df=char_bos if "char_bos" in globals() else char,
                  colloc_df=colloc)

Saved 8 head rows to /Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-head-characterization.csv
Saved 102 collocation rows to /Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-collocation.csv


{'characterization': PosixPath('/Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-head-characterization.csv'),
 'collocation': PosixPath('/Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-collocation.csv')}

## Delete Model

In [14]:
# Run this between models
import gc
import torch

for _name in ["model", "cache", "char", "colloc"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("Memory cleared")

Memory cleared
